# Train a Regression Model

In this notebook we train a linear regression model on the NYC Green Taxi dataset and track the run with MLflow. The goal is not to build the most complex model, but to create a clean baseline that we can later monitor for drift.

We will use one month of data, a small feature set, and a local MLflow tracking store so the rest of the repository can stay runnable on a single machine.


In [ ]:
import os
from pathlib import Path
from urllib.request import urlretrieve

from dotenv import load_dotenv

import pandas as pd

import mlflow
from mlflow import MlflowClient

from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split

In [ ]:
year = 2024
month = 1
color = "green"

In [ ]:
# Download the data
data_path = Path(f"./data/{color}_tripdata_{year}-{month:02d}.parquet")
data_path.parent.mkdir(exist_ok=True)

if not data_path.exists():
    urlretrieve(
        f"https://d37ci6vzurychx.cloudfront.net/trip-data/{color}_tripdata_{year}-{month:02d}.parquet",
        data_path,
    )

In [ ]:
# Load the data
df = pd.read_parquet(f"./data/{color}_tripdata_{year}-{month:02d}.parquet")

In [ ]:
df.shape

Now we will connect the notebook to MLflow. For a local-first setup, copy `.env.example` to `.env` and keep `MLFLOW_TRACKING_URI=./mlruns` unless you already have an MLflow server running elsewhere.

You can also start the MLflow UI from the repository root with:

```bash
mlflow ui --backend-store-uri ./mlruns --port 5000
```


We will create or reuse an experiment called `green-taxi-monitoring` in MLflow. We also add a small tag set so the training run is easier to identify later when we load it in the monitoring chapters.


In [ ]:
# Load environment variables from .env file
load_dotenv()

# Get the MLflow tracking URI from environment variables
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "./mlruns")

# Set up the connection to MLflow
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

experiment_name = "green-taxi-monitoring"
exp = mlflow.get_experiment_by_name(experiment_name)

if exp is None:
    experiment_id = mlflow.create_experiment(
        name=experiment_name,
        tags={"developer": "<your-name>"},
    )
    print(f"Created experiment: {experiment_name} (ID: {experiment_id})")
else:
    print(f"Using existing experiment: {experiment_name} (ID: {exp.experiment_id})")

mlflow.set_experiment(experiment_name)

If everything went well, you should now be able to inspect the experiment in the MLflow UI. With the local setup above, that UI is available at `http://127.0.0.1:5000`.


Let's start now by looking at the data:

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
# Look for missing values
df.isnull().sum()

Nearly all selected features have useful values for this first baseline. To predict the duration of a trip, we will use:

- `PULocationID`: The pickup location ID
- `DOLocationID`: The dropoff location ID
- `trip_distance`: The distance of the trip in miles
- `passenger_count`: The number of passengers
- `fare_amount`: The fare amount in USD
- `total_amount`: The total amount in USD

First we calculate trip duration in minutes from the Green Taxi pickup and dropoff timestamps: `lpep_pickup_datetime` and `lpep_dropoff_datetime`. We also remove trips shorter than 1 minute or longer than 60 minutes so the baseline is not dominated by obvious outliers.

In [ ]:
features = [
    "PULocationID",
    "DOLocationID",
    "trip_distance",
    "passenger_count",
    "fare_amount",
    "total_amount",
]
target = "duration"

In [ ]:
# Calculate duration and keep trips in a practical 1 to 60 minute range.
def calculate_trip_duration_in_minutes(df):
    df["duration"] = (
        df["lpep_dropoff_datetime"] - df["lpep_pickup_datetime"]
    ).dt.total_seconds() / 60
    df = df[(df["duration"] >= 1) & (df["duration"] <= 60)]
    df = df[(df["passenger_count"] > 0) & (df["passenger_count"] < 8)]
    df = df[features + [target]]
    return df

In [ ]:
df_processed = calculate_trip_duration_in_minutes(df)

In [ ]:
df_processed.head()

In [ ]:
df_processed.duration.describe()

In [ ]:
df_processed.duration.hist()

Now that we have the dataframe that we want to train our model on, we need to split it into a train and test set. We will use 80% of the data for training and 20% for testing.

In [ ]:
y = df_processed["duration"]
X = df_processed.drop(columns=["duration"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, random_state=42, test_size=0.2
)

And now we can train the model and track the run with MLflow. We will set a few tags to make the run easier to find later.

- `model`: `linear regression`
- `dataset`: `green-taxi`
- `developer`: `your name`
- `features`: the columns used for training
- `target`: the value we want to predict
- `year` and `month`: which batch produced the training data

We will also log the evaluation metric and register the trained model under a stable name so the next notebooks can load it again.


In [ ]:
# No extra cloud credentials are needed when you use a local MLflow tracking store.
# The model artifact will be stored wherever MLFLOW_TRACKING_URI points.


In [ ]:
with mlflow.start_run():
    tags = {
        "model": "linear regression",
        "developer": "<your-name>",
        "dataset": f"{color}-taxi",
        "year": year,
        "month": month,
        "features": features,
        "target": target,
    }
    mlflow.set_tags(tags)

    lr = LinearRegression()
    lr.fit(X_train, y_train)

    y_pred = lr.predict(X_test)
    rmse = root_mean_squared_error(y_test, y_pred)
    mlflow.log_metric("rmse", rmse)
    mlflow.sklearn.log_model(sk_model=lr, name="mlflow-model")
    active_run = mlflow.active_run()
    assert active_run is not None
    run_id = active_run.info.run_id

    client = MlflowClient()
    model_name = "green-taxi-ride-duration"
    model_uri = f"runs:/{run_id}/mlflow-model"

    try:
        client.create_registered_model(
            name=model_name,
            tags={"framework": "scikit-learn"},
            description="Model to predict ride duration in NYC",
        )
    except Exception:
        print(f"Registered model {model_name} already exists. Reusing it.")

    mv = client.create_model_version(
        name=model_name,
        source=model_uri,
        run_id=run_id,
    )

    client.set_registered_model_alias(
        name=model_name,
        version=mv.version,
        alias="production",
    )

You should now see your run in the MLflow UI. Under the created experiment, you can also see the logged tags, the metric and the saved model.

![](images/monitoring_mlflow_registered_model.png)

And you can see what you need to do to load the model in an API or script in the UI as long as the application has access to MLflow.